## Verifica di funzioni sul dataframe

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Carica il dataset originale
df = pd.read_excel("data/data-ready-def.xlsx")

In [2]:
df.shape[0] # numero di righe dataframe

3519

In [3]:
df.columns # elenco delle colonne del dataframe

Index(['Date', 'Air pressure on the ground(hPa)', 'Average temperatures(¡æ)',
       'Maximum temperature(¡æ)', 'Minimum temperature(¡æ)',
       'Precipitation(mm)', 'Relative humidity(%)', 'Wind velocity(m/s)',
       'Hours of sunshine(h)', 'ET0', 'Water demand', 'Amount of irrigation'],
      dtype='str')

In [4]:
#aggiungo la colonna 'year' al dataframe, che contiene l'anno estratto dalla colonna 'Date', 13 colonne in totale
df['year'] = df['Date'] //10000  # estrae l'anno dalla data (prime 4 cifre)

In [5]:
df.groupby('year').size() # numero di righe per ogni anno

year
2000    153
2001    153
2002    153
2003    153
2004    153
2005    153
2006    153
2007    153
2008    153
2009    153
2010    153
2011    153
2012    153
2013    153
2014    153
2015    153
2016    153
2017    153
2018    153
2019    153
2020    153
2021    153
2022    153
dtype: int64

### Slicing

In [6]:
#training set
train_df = df[df['year'] <= 2016].reset_index(drop=True) #primi 17 anni del dataframe (dal 2000 al 2016, 17 anni * 153 righe/anno = 2601 righe)


In [7]:
#validation set
valid_df = df[(df['year'] >= 2017) & (df['year'] <= 2019)].reset_index(drop=True) # secondi 3 anni (dal 2017 al 2019, 3 anni * 153 righe/anno = 459 righe) 

In [8]:
#test set
test_df = df[df['year'] >= 2020].reset_index(drop=True) # ultimi 3 anni (dal 2020 al 2022, 3 anni * 153 righe/anno = 459 righe) 

In [9]:
#verifica slicing corretto
print(train_df.shape, train_df['year'].min(), train_df['year'].max()) # 2601 righe, 13 colonne
print(valid_df.shape, valid_df['year'].min(), valid_df['year'].max()) # 459 righe, 13 colonne
print(test_df.shape, test_df['year'].min(), test_df['year'].max()) # 459 righe, 13 colonne

(2601, 13) 2000 2016
(459, 13) 2017 2019
(459, 13) 2020 2022


In [10]:
#rimuovo le colonne 'ET0' e 'Water demand' dai dataframe train, valid e test
train_df_clean = train_df.drop(columns=['ET0', 'Water demand'])
valid_df_clean = valid_df.drop(columns=['ET0', 'Water demand'])
test_df_clean = test_df.drop(columns=['ET0', 'Water demand'])

In [11]:
(df['Amount of irrigation'] < 0).sum() # numero di righe con valori negativi nella colonna 'Amount of irrigation' del dataframe originale

np.int64(917)

In [12]:
#sostituisco i valori negativi con 0 nella colonna 'Amount of irrigation' dei 3 dataframe modificati
train_df_clean['Amount of irrigation'] = train_df_clean['Amount of irrigation'].clip(lower=0)
valid_df_clean['Amount of irrigation'] = valid_df_clean['Amount of irrigation'].clip(lower=0)
test_df_clean['Amount of irrigation'] = test_df_clean['Amount of irrigation'].clip(lower=0)


In [13]:
#verifica pulizia dataframes, due colonne in meno e nessun valore negativo nella colonna 'Amount of irrigation'
print(train_df_clean.shape)
print(valid_df_clean.shape)
print(test_df_clean.shape)
print((train_df_clean['Amount of irrigation'] < 0).sum())
print((valid_df_clean['Amount of irrigation'] < 0).sum())
print((test_df_clean['Amount of irrigation'] < 0).sum())

(2601, 11)
(459, 11)
(459, 11)
0
0
0


### Normalizzazione

In [14]:
colonne = train_df_clean.columns.drop(['Date', 'year']) #nomi delle colonne escluse Date e year

#calcolo il minimo e il massimo per ogni colonna del dataframe aggiornato senza le due colonne
train_df_min = train_df_clean[colonne].min()
train_df_max = train_df_clean[colonne].max()
#questi valori mi serviranno anche per la normalizzazione inversa, li salvo.

print("Minimi:")
print(train_df_min)

print("\nMassimi:")
print(train_df_max)

Minimi:
Air pressure on the ground(hPa)    958.89
Average temperatures(¡æ)             1.69
Maximum temperature(¡æ)              6.67
Minimum temperature(¡æ)             -2.35
Precipitation(mm)                    0.00
Relative humidity(%)                17.65
Wind velocity(m/s)                   0.67
Hours of sunshine(h)                 0.44
Amount of irrigation                 0.00
dtype: float64

Massimi:
Air pressure on the ground(hPa)    1004.310
Average temperatures(¡æ)             31.700
Maximum temperature(¡æ)              38.050
Minimum temperature(¡æ)              26.360
Precipitation(mm)                   121.960
Relative humidity(%)                 96.690
Wind velocity(m/s)                    9.960
Hours of sunshine(h)                  8.230
Amount of irrigation                 12.236
dtype: float64


In [15]:
#normalizzo i valori dei tre dataset utilizzando i minimi e i massimi calcolati sul solo training set, per evitare data leakage

train_df_clean[colonne] = (train_df_clean[colonne] - train_df_min) / (train_df_max - train_df_min)
valid_df_clean[colonne] = (valid_df_clean[colonne] - train_df_min) / (train_df_max - train_df_min)
test_df_clean[colonne] = (test_df_clean[colonne] - train_df_min) / (train_df_max - train_df_min)

#verifico che i valori del train siano compresi tra 0 e 1 dopo la normalizzazione
train_df_min_check = train_df_clean[colonne].min()
train_df_max_check = train_df_clean[colonne].max()

#questi mi aspetto che non lo siano tutti
valid_df_min = valid_df_clean[colonne].min()
valid_df_max = valid_df_clean[colonne].max()

test_df_min = test_df_clean[colonne].min()
test_df_max = test_df_clean[colonne].max()

print("Minimi train:")
print(train_df_min_check)
print("\nMassimi train:")
print(train_df_max_check)

print("Minimi valid:")
print(valid_df_min)
print("\nMassimi valid:")
print(valid_df_max)

print("Minimi test:")
print(test_df_min)
print("\nMassimi test:")
print(test_df_max)

Minimi train:
Air pressure on the ground(hPa)    0.0
Average temperatures(¡æ)           0.0
Maximum temperature(¡æ)            0.0
Minimum temperature(¡æ)            0.0
Precipitation(mm)                  0.0
Relative humidity(%)               0.0
Wind velocity(m/s)                 0.0
Hours of sunshine(h)               0.0
Amount of irrigation               0.0
dtype: float64

Massimi train:
Air pressure on the ground(hPa)    1.0
Average temperatures(¡æ)           1.0
Maximum temperature(¡æ)            1.0
Minimum temperature(¡æ)            1.0
Precipitation(mm)                  1.0
Relative humidity(%)               1.0
Wind velocity(m/s)                 1.0
Hours of sunshine(h)               1.0
Amount of irrigation               1.0
dtype: float64
Minimi valid:
Air pressure on the ground(hPa)    0.209159
Average temperatures(¡æ)           0.133955
Maximum temperature(¡æ)            0.092097
Minimum temperature(¡æ)            0.112504
Precipitation(mm)                  0.000000
Rela

### Splitting years

In [20]:
#salvo le colonne di features
feature_columns = [c for c in train_df_clean.columns if c not in ['Date', 'year', 'Amount of irrigation']]
target_column = 'Amount of irrigation'  #salvo il nome della colonna target

look_back = 20
H = 5

X_list = []
Y_list = []

for year in train_df_clean['year'].unique():
  year_data = train_df_clean[train_df_clean['year'] == year]
  #print(year, year_data.shape)

  #con values i due tipi di dati sono sotto forma di array numpy, numerato con indici 0, 1, 2...
  features = year_data[feature_columns].values  #shape (153, 8)
  target = year_data[target_column].values  #shape (153,)

  for i in range(len(year_data) - look_back - H + 1):
    window = features[i:i + look_back]
    target_value = target[i + look_back - 1 + H]

    X_list.append(window)
    Y_list.append(target_value)

 
X = np.array(X_list)
Y = np.array(Y_list)

print(X.shape)
print(Y.shape)


(2193, 20, 8)
(2193,)
